<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/NMT_project_Darrick_Pang_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import time

In [2]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00


In [3]:
pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.7 MB/s eta 0:00:00


In [4]:
from evaluate import load
bleu = load("sacrebleu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(100000))
test_data = dataset["test"]
val_data = dataset["validation"]

README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/282M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/277M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/475k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4548885 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2169 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2999 [00:00<?, ? examples/s]

In [6]:
print(train_data["translation"])
print(val_data)

Column([{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}, {'de': 'Ich erkläre die am Freitag, dem 17. Dezember unterbrochene Sitzungsperiode des Europäischen Parlaments für wiederaufgenommen, wünsche Ihnen nochmals alles Gute zum Jahreswechsel und hoffe, daß Sie schöne Ferien hatten.', 'en': 'I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.'}, {'de': 'Wie Sie feststellen konnten, ist der gefürchtete "Millenium-Bug " nicht eingetreten. Doch sind Bürger einiger unserer Mitgliedstaaten Opfer von schrecklichen Naturkatastrophen geworden.', 'en': "Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series of natural disasters that truly were dreadful."}, {'de': 'Im Parlament besteht der Wunsch nach einer Aussprache im V

In [7]:
source_text = [german["de"] for german in train_data["translation"]]
# print(source_text)

target_text = [english["en"] for english in train_data["translation"]]
# print(target_text)

# Add start and end tokens to target text
target_text_with_tokens = ['<start> ' + text + ' <end>' for text in target_text]


source_tokenizer = Tokenizer(num_words=30000, filters='')
target_tokenizer = Tokenizer(num_words=30000, filters='')

source_tokenizer.fit_on_texts(source_text)
target_tokenizer.fit_on_texts(target_text_with_tokens)

source_sequence = source_tokenizer.texts_to_sequences(source_text)
target_sequence = target_tokenizer.texts_to_sequences(target_text_with_tokens)

max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')

In [8]:
source_vocab = 30000
target_vocab = 30000
embedding_dim = 256
latent_dim = 512

encoder_inputs = layers.Input(shape=(max_src_len,))
encoder_embeddings = layers.Embedding(source_vocab, embedding_dim)(encoder_inputs)
encoder_lstm = layers.Bidirectional(layers.LSTM(latent_dim, return_sequences=True, return_state=True))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_embeddings)

# Concatenate forward and backward states
state_h = layers.Concatenate()([forward_h, backward_h])
state_c = layers.Concatenate()([forward_c, backward_c])


decoder_inputs = layers.Input(shape=(1,))
decoder_embeddings = layers.Embedding(target_vocab, embedding_dim)(decoder_inputs)
decoder_lstm = layers.LSTM(latent_dim * 2, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embeddings, initial_state=[state_h, state_c])

attention = layers.Attention()([decoder_outputs, encoder_outputs])
decoder_concat = layers.Concatenate(axis=-1)([decoder_outputs, attention])

# Add a Dense layer for outputting probabilities for each word in the target vocabulary
decoder_dense = layers.Dense(target_vocab, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)


model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [9]:
start_time = time.time()

model.fit([encoder_input, decoder_input], np.expand_dims(decoder_target, -1), batch_size=32, epochs=1, validation_split=0.1)

end_time = time.time()
elapsed = end_time - start_time
print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

2813/2813 ━━━━━━━━━━━━━━━━━━━━ 1054s 373ms/step - accuracy: 0.5114 - loss: 3.5069 - val_accuracy: 0.6009 - val_loss: 2.5469
Training time: 1054.22 seconds (17.57 minutes)


In [11]:
encoder_inference_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

decoder_state_input_h = layers.Input(shape=(latent_dim * 2,)) # Corrected shape
decoder_state_input_c = layers.Input(shape=(latent_dim * 2,)) # Corrected shape
encoder_outputs_input = layers.Input(shape=(max_src_len, latent_dim*2)) # Input for encoder outputs, shape should match encoder_outputs


decoder_output, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embeddings, initial_state=[decoder_state_input_h, decoder_state_input_c]
)

attention_inf = layers.Attention()([decoder_output, encoder_outputs_input])
decoder_concat_inf = layers.Concatenate(axis=-1)([decoder_output, attention_inf])


decoder_output_inf = decoder_dense(decoder_concat_inf)

decoder_model = Model(
    [decoder_inputs, decoder_state_input_h, decoder_state_input_c, encoder_outputs_input],
    [decoder_output_inf, state_h_inf, state_c_inf]
)

In [12]:
def translate(sentence):
  seq = source_tokenizer.texts_to_sequences([sentence])
  seq = pad_sequences(seq, maxlen=max_src_len, padding='post')
  encoder_outputs_inf, h, c = encoder_inference_model.predict(seq)

  target_seq = np.array([[target_tokenizer.word_index['<start>']]])

  result = ''

  for _ in range(max_tgt_len):
        # Explicitly reshape target_seq to have a sequence length dimension
        target_seq_reshaped = np.reshape(target_seq, (target_seq.shape[0], 1, target_seq.shape[1]))
        output, h, c = decoder_model.predict([target_seq_reshaped, h, c, encoder_outputs_inf])
        token = np.argmax(output[0, -1, :])
        word = target_tokenizer.index_word.get(token, '')
        if word == '<end>':
            break
        result += ' ' + word
        target_seq = np.array([[token]])
  return result.strip()

In [13]:
print(translate("Das ist ein Test."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
that is a


In [14]:
# Generate predictions on a small subset of validation data
predictions = []
references = []

for i in range(2):  # 200 sentences for demo, increase later
    de_sentence = test_data[i]["translation"]["de"]
    en_reference = test_data[i]["translation"]["en"]

    en_predicted = translate(de_sentence)

    predictions.append(en_predicted)
    references.append([en_reference])  # sacreBLEU expects list of list

results = bleu.compute(predictions=predictions, references=references)
print(f"BLEU score: {results['score']:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
BLEU score: 6.34


In [ ]:
import nbformat

input_path = "NMT_project_Darrick_Pang_original.ipynb"
output_path = "NMT_project_Darrick_Pang.ipynb"

with open(input_path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# Remove Colab metadata while keeping outputs
for cell in nb.cells:
    if "colab" in cell.metadata:
        del cell.metadata["colab"]
    if "id" in cell.metadata:
        del cell.metadata["id"]
if "colab" in nb.metadata:
    del nb.metadata["colab"]

with open(output_path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print(f"✅ Cleaned notebook saved as {output_path}")
